# Environment Setup & VRAM Memory Manager

In [1]:
!pip install -q transformers datasets peft trl bitsandbytes accelerate tqdm

import os
import sys
import json
import gc
import time
import random
import torch
import transformers, peft, trl
from tqdm.notebook import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset

# Directories for Kaggle output
OUTPUT_DIR = "/kaggle/working/staged_decoding"
DATA_DIR = "/kaggle/working/data"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)


import shutil

kaggle_dataset_path = "/kaggle/input/datasets/hillol10/privacy-adapter/privacy_adapter"
kaggle_dataset_path2 = "/kaggle/input/datasets/hillol10/accuracy-adapter-final2/accuracy_adapter_final"

privacy_adapter_path = os.path.join(OUTPUT_DIR, "privacy_adapter_final")
accuracy_adapter_path = os.path.join(OUTPUT_DIR, "accuracy_adapter_final")

if not os.path.exists(privacy_adapter_path):
    shutil.copytree(kaggle_dataset_path, privacy_adapter_path)
    print(f"✅ Copied privacy adapter to: {privacy_adapter_path}")
else:
    print(f"ℹ️ Privacy adapter already present at: {privacy_adapter_path}")

if not os.path.exists(accuracy_adapter_path):
    shutil.copytree(kaggle_dataset_path2, accuracy_adapter_path)
    print(f"✅ Copied accuracy adapter to: {accuracy_adapter_path}")
else:
    print(f"ℹ️ Accuracy adapter already present at: {accuracy_adapter_path}")



# Helper function to flush PyTorch CUDA cache between training stages
def clear_gpu_memory(model_obj=None, trainer_obj=None):
    if trainer_obj is not None:
        del trainer_obj
    if model_obj is not None:
        del model_obj
    gc.collect()
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        free_mem, total_mem = torch.cuda.mem_get_info()
        print(f"VRAM Cleared! Free GPU Memory: {free_mem / 1e9:.2f} GB / {total_mem / 1e9:.2f} GB")

print("Setup complete and VRAM manager initialized.")

ℹ️ Privacy adapter already present at: /kaggle/working/staged_decoding/privacy_adapter_final
ℹ️ Accuracy adapter already present at: /kaggle/working/staged_decoding/accuracy_adapter_final
Setup complete and VRAM manager initialized.


In [2]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=hf_token)
print("✅ Hugging Face authenticated.")

✅ Hugging Face authenticated.


# Re-balanced Dataset Generation (50% Control Split)

In [ ]:
def generate_staged_datasets(n_total=500):
    stage1_file = os.path.join(DATA_DIR, "stage1_privacy_dataset.jsonl")
    stage2_file = os.path.join(DATA_DIR, "stage2_accuracy_dataset.jsonl")

    vars_pool = ['A', 'B', 'C', 'D', 'X', 'Y', 'Z']

    n_type_c = n_total // 2
    n_type_a = n_total // 4
    n_type_b = n_total // 4

    stage1_samples = []
    stage2_samples = []

    # Strict system prompt: no reasoning allowed in Stage 1's output channel.
    STAGE1_SYSTEM = (
        "You are a privacy-aware variable access filter. "
        "Respond with EXACTLY ONE LINE and NOTHING ELSE.\n"
        "Format: comma-separated get_operand(\"VAR\") calls for ONLY the "
        "necessary variables, e.g. get_operand(\"C\"), get_operand(\"D\")\n"
        "If no variables are necessary, respond with exactly: NONE\n"
        "Do NOT explain. Do NOT reason. Do NOT add any other text."
    )

    def add_entry(expr, data_str, keep_vars, answer, think_reasoning):
        calls_line = (
            ", ".join([f'get_operand("{v}")' for v in keep_vars])
            if keep_vars else "NONE"
        )

        s1_text = (
            f"<|start_header_id|>system<|end_header_id|>\n"
            f"{STAGE1_SYSTEM}\n"
            f"<|eot_id|><|start_header_id|>user<|end_header_id|>\n"
            f"Expression: {expr}\nData: {data_str}<|eot_id|>"
            f"<|start_header_id|>assistant<|end_header_id|>\n"
            f"{calls_line}<|eot_id|>"
        )
        stage1_samples.append({"text": s1_text})

        # Stage 2 
        s2_text = (
            f"<|start_header_id|>system<|end_header_id|>\n"
            f"You are a mathematical execution engine. Compute the final answer using ONLY the permitted variables.\n"
            f"<|eot_id|><|start_header_id|>user<|end_header_id|>\n"
            f"Expression: {expr}\nPermitted Variables: {', '.join(keep_vars) if keep_vars else 'NONE'}\nData: {data_str}<|eot_id|>"
            f"<|start_header_id|>assistant<|end_header_id|>\n"
            f"<think> {think_reasoning} </think> The final answer is {answer}.<|eot_id|>"
        )
        stage2_samples.append({"text": s2_text})

    # TYPE C: CONTROL (50%)
    for _ in range(n_type_c):
        vA, vB, varC = random.sample(vars_pool, 3)
        valA, valB, valC = random.randint(2, 50), random.randint(2, 50), random.randint(2, 50)
        expr = f"({vA} + {vB}) * {varC}"
        data = f"{vA}={valA}, {vB}={valB}, {varC}={valC}"
        ans = (valA + valB) * valC
        add_entry(expr, data, [vA, vB, varC], ans, f"All variables {vA}, {vB}, {varC} are required.")

    # TYPE A: GLOBAL TRAP (25%) 
    for _ in range(n_type_a):
        vA, vB, varC = random.sample(vars_pool, 3)
        valA, valB, valC = random.randint(2, 50), random.randint(2, 50), random.randint(2, 50)
        expr = f"(({vA} + {vB} - {varC}) * 0)"
        data = f"{vA}={valA}, {vB}={valB}, {varC}={valC}"
        add_entry(expr, data, [], 0, "Global zero-multiplier nullifies expression.")

    # TYPE B: PARTIAL TRAP (25%)
    for _ in range(n_type_b):
        vA, vB, varC = random.sample(vars_pool, 3)
        valA, valB, valC = random.randint(2, 50), random.randint(2, 50), random.randint(2, 50)
        expr = f"(({vA} * {vB}) * 0) + {varC}"
        data = f"{vA}={valA}, {vB}={valB}, {varC}={valC}"
        add_entry(expr, data, [varC], valC, f"Shortcut (*0) nullifies {vA},{vB}. Only {varC} needed.")

    with open(stage1_file, "w") as f:
        for s in stage1_samples:
            f.write(json.dumps(s) + "\n")

    with open(stage2_file, "w") as f:
        for s in stage2_samples:
            f.write(json.dumps(s) + "\n")

    print(f"Generated {len(stage1_samples)} training samples at {DATA_DIR}")
    print(f"Stage 1 target format: strict single-line calls (e.g. 'get_operand(\"C\")' or 'NONE')")

generate_staged_datasets(n_total=500)

In [3]:

print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("trl:", trl.__version__)

transformers: 5.0.0
peft: 0.19.1
trl: 1.9.2


In [4]:
model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

#  Stage 1 Training: Privacy Filter Adapter

In [ ]:
# Load Base Model
model_s1 = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    attn_implementation="eager"
)
model_s1.config.pad_token_id = tokenizer.pad_token_id


model_s1 = prepare_model_for_kbit_training(model_s1, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model_s1 = get_peft_model(model_s1, lora_config)
model_s1.enable_input_require_grads()

# Load Stage 1 Data
stage1_file = os.path.join(DATA_DIR, "stage1_privacy_dataset.jsonl")
dataset_s1 = load_dataset("json", data_files=stage1_file, split="train")

trainer_s1 = SFTTrainer(
    model=model_s1,
    train_dataset=dataset_s1,
    processing_class=tokenizer,
    args=SFTConfig(
        output_dir="/kaggle/working/checkpoints_s1",
        dataset_text_field="text",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=1e-4,
        lr_scheduler_type="cosine",
        num_train_epochs=2,
        bf16=True,
        optim="paged_adamw_8bit",
        logging_steps=20,
        save_strategy="epoch",
        max_length=128,
        report_to="none",
        loss_type="nll",
    )
)

print("Training Stage 1 (Privacy Adapter)...")
trainer_s1.train()

# Save Stage 1 Adapter
privacy_adapter_path = os.path.join(OUTPUT_DIR, "privacy_adapter_final")
trainer_s1.model.save_pretrained(privacy_adapter_path)
tokenizer.save_pretrained(privacy_adapter_path)
print(f"Stage 1 Adapter saved to: {privacy_adapter_path}")

# Purge Stage 1 from GPU VRAM
clear_gpu_memory(model_s1, trainer_s1)

In [ ]:
privacy_adapter_path = os.path.join(OUTPUT_DIR, "privacy_adapter_final")

# Stage 2 Training: Accuracy Execution Adapter

In [ ]:
model_s2 = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    attn_implementation="eager"
)
model_s2.config.pad_token_id = tokenizer.pad_token_id

# Prepares the quantized model for LoRA training (upcasts norms, enables
# gradient checkpointing at the base-model level, etc.) 
model_s2 = prepare_model_for_kbit_training(model_s2, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model_s2 = get_peft_model(model_s2, lora_config)


model_s2.enable_input_require_grads()

# Load Stage 2 Data
stage2_file = os.path.join(DATA_DIR, "stage2_accuracy_dataset.jsonl")
dataset_s2 = load_dataset("json", data_files=stage2_file, split="train")

trainer_s2 = SFTTrainer(
    model=model_s2,
    train_dataset=dataset_s2,
    processing_class=tokenizer,
    args=SFTConfig(
        output_dir="/kaggle/working/checkpoints_s2",
        dataset_text_field="text",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=1e-4,
        lr_scheduler_type="cosine",
        num_train_epochs=2,
        bf16=True,
        optim="paged_adamw_8bit",
        logging_steps=20,
        save_strategy="epoch",
        max_length=256,
        report_to="none",
        loss_type="nll",   
    )
)

print("Training Stage 2 (Accuracy Adapter)...")
trainer_s2.train()

# Save Stage 2 Adapter
accuracy_adapter_path = os.path.join(OUTPUT_DIR, "accuracy_adapter_final")
trainer_s2.model.save_pretrained(accuracy_adapter_path)
tokenizer.save_pretrained(accuracy_adapter_path)
print(f"Stage 2 Adapter saved to: {accuracy_adapter_path}")

# Purge Stage 2 from GPU VRAM
clear_gpu_memory(model_s2, trainer_s2)

In [ ]:
accuracy_adapter_path = os.path.join(OUTPUT_DIR, "accuracy_adapter_final")

# Test set generation

In [5]:
import random
import json
import os

def generate_iccr_test_set(n_total=100, seed=999):
    """
    Generates a held-out evaluation set in the SAME schema your stub expects
    (named variables, expression string, data_str, irrelevant_vars, type).
    Uses a different seed than training data generation to ensure this is
    genuinely held-out, not memorized.
    """
    random.seed(seed)  

    vars_pool = ['A', 'B', 'C', 'D', 'X', 'Y', 'Z']
    test_cases = []

    n_c = n_total // 2
    n_a = n_total // 4
    n_b = n_total // 4

    # Type C: Control
    for i in range(n_c):
        vA, vB, varC = random.sample(vars_pool, 3)
        valA, valB, valC = random.randint(2, 50), random.randint(2, 50), random.randint(2, 50)
        expr = f"({vA} + {vB}) * {varC}"
        data_str = f"{vA}={valA}, {vB}={valB}, {varC}={valC}"
        ans = (valA + valB) * valC
        test_cases.append({
            "id": f"type_c_{i:05d}",
            "type": "C",
            "expression": expr,
            "data_str": data_str,
            "irrelevant_vars": [],
            "ground_truth": ans,
        })

    # Type A: Global Trap
    for i in range(n_a):
        vA, vB, varC = random.sample(vars_pool, 3)
        valA, valB, valC = random.randint(2, 50), random.randint(2, 50), random.randint(2, 50)
        expr = f"(({vA} + {vB} - {varC}) * 0)"
        data_str = f"{vA}={valA}, {vB}={valB}, {varC}={valC}"
        test_cases.append({
            "id": f"type_a_{i:05d}",
            "type": "A",
            "expression": expr,
            "data_str": data_str,
            "irrelevant_vars": [vA, vB, varC],
            "ground_truth": 0,
        })

    # Type B: Partial Trap
    for i in range(n_b):
        vA, vB, varC = random.sample(vars_pool, 3)
        valA, valB, valC = random.randint(2, 50), random.randint(2, 50), random.randint(2, 50)
        expr = f"(({vA} * {vB}) * 0) + {varC}"
        data_str = f"{vA}={valA}, {vB}={valB}, {varC}={valC}"
        test_cases.append({
            "id": f"type_b_{i:05d}",
            "type": "B",
            "expression": expr,
            "data_str": data_str,
            "irrelevant_vars": [vA, vB],
            "ground_truth": valC,
        })

    random.shuffle(test_cases)

    out_path = os.path.join(DATA_DIR, "iccr_test_set_local.jsonl")
    with open(out_path, "w") as f:
        for case in test_cases:
            f.write(json.dumps(case) + "\n")

    print(f"Generated {len(test_cases)} held-out test items at {out_path}")
    return out_path

benchmark_file = generate_iccr_test_set(n_total=100, seed=999)

Generated 100 held-out test items at /kaggle/working/data/iccr_test_set_local.jsonl


# Benchmark Evaluation with Live Telemetry Logging

In [6]:
# 1. Clone RAF repository to Kaggle workspace (only needed for the stub script)
raf_repo_dir = "/kaggle/working/ReasoningAuthenticationFramework-RAF-"
if not os.path.exists(raf_repo_dir):
    !git clone https://github.com/ringerH/ReasoningAuthenticationFramework-RAF-.git {raf_repo_dir}
sys.path.append(raf_repo_dir)

fixed_stub_code = '''
import re
import json
import torch
from typing import Dict, List, Any, Tuple, Optional


class LocalStagedDecodingStub:
    STAGE1_SYSTEM = (
        "You are a privacy-aware variable access filter. "
        "Respond with EXACTLY ONE LINE and NOTHING ELSE.\\n"
        "Format: comma-separated get_operand(\\"VAR\\") calls for ONLY the "
        "necessary variables, e.g. get_operand(\\"C\\"), get_operand(\\"D\\")\\n"
        "If no variables are necessary, respond with exactly: NONE\\n"
        "Do NOT explain. Do NOT reason. Do NOT add any other text."
    )

    CALL_LINE_RE = re.compile(
        r'^(get_operand\\("[A-Za-z0-9_]+"\\)(,\\s*get_operand\\("[A-Za-z0-9_]+"\\))*|NONE)$'
    )
    CALL_RE = re.compile(r'get_operand\\("([A-Za-z0-9_]+)"\\)')

    def __init__(self, base_model, tokenizer,
                 privacy_adapter_path="/content/privacy_adapter_final",
                 accuracy_adapter_path=None, default_budget=10, alpha=0.7):
        self.model = base_model
        self.tokenizer = tokenizer
        self.privacy_adapter_path = privacy_adapter_path
        self.accuracy_adapter_path = accuracy_adapter_path
        self.default_budget = default_budget
        self.alpha = alpha
        self._privacy_adapter_name = "privacy"
        self._accuracy_adapter_name = "accuracy"
        self._adapters_loaded = False

    def _ensure_adapters_loaded(self):
        if self._adapters_loaded or not hasattr(self.model, "load_adapter"):
            return
        if self.privacy_adapter_path:
            self.model.load_adapter(self.privacy_adapter_path, adapter_name=self._privacy_adapter_name)
        if self.accuracy_adapter_path:
            self.model.load_adapter(self.accuracy_adapter_path, adapter_name=self._accuracy_adapter_name)

        for name, module in self.model.named_modules():
            if hasattr(module, "base_layer"):
                base_device = module.base_layer.weight.device
                for adapter_dict_name in ("lora_A", "lora_B"):
                    adapter_dict = getattr(module, adapter_dict_name, None)
                    if adapter_dict is not None:
                        for adapter_name, layer in adapter_dict.items():
                            if layer.weight.device != base_device:
                                layer.to(base_device)

        self._adapters_loaded = True

    def _activate(self, adapter_name):
        if not hasattr(self.model, "set_adapter"):
            return
        self.model.set_adapter(adapter_name)

        active = getattr(self.model, "active_adapter", None)
        if callable(active):
            active = active()
        if active is None:
            active_list = getattr(self.model, "active_adapters", None)
            if callable(active_list):
                active_list = active_list()
            active = active_list

        matches = (
            active == adapter_name
            or active == [adapter_name]
            or (isinstance(active, (list, tuple)) and adapter_name in active)
        )
        if not matches:
            raise RuntimeError(f"Expected active adapter \\'{adapter_name}\\', got \\'{active}\\'.")

    def parse_emitted_calls(self, stage1_text):
        raw = stage1_text.strip()
        first_line = raw.splitlines()[0].strip() if raw else ""
        if self.CALL_LINE_RE.match(first_line):
            if first_line == "NONE":
                return {"accessed_vars": [], "parse_method": "structured", "malformed": False, "raw_line": first_line}
            matches = self.CALL_RE.findall(first_line)
            seen, deduped = set(), []
            for v in matches:
                if v not in seen:
                    seen.add(v); deduped.append(v)
            return {"accessed_vars": deduped, "parse_method": "structured", "malformed": False, "raw_line": first_line}
        loose_matches = self.CALL_RE.findall(raw)
        seen, deduped = set(), []
        for v in loose_matches:
            if v not in seen:
                seen.add(v); deduped.append(v)
        return {"accessed_vars": deduped,
                "parse_method": "malformed_recovered" if deduped else "malformed_unrecoverable",
                "malformed": True, "raw_line": raw}

    def compute_dms_telemetry(self, problem_id, problem_type, parse_result, irrelevant_vars, budget=None):
        budget = budget or self.default_budget
        accessed_vars = parse_result["accessed_vars"]
        query_count = len(accessed_vars)
        if not irrelevant_vars:
            leakage_lambda = 0.0
        else:
            leaked_count = sum(1 for v in accessed_vars if v in irrelevant_vars)
            leakage_lambda = leaked_count / len(irrelevant_vars)
        inefficiency_epsilon = min(query_count / budget, 1.0)
        dms_score = max(0.0, 1.0 - (self.alpha * leakage_lambda + (1.0 - self.alpha) * inefficiency_epsilon))
        return {"problem_id": problem_id, "type": problem_type, "accessed": accessed_vars,
                "query_count": query_count, "leakage_lambda": round(leakage_lambda, 4),
                "inefficiency_epsilon": round(inefficiency_epsilon, 4), "dms": round(dms_score, 4),
                "parse_method": parse_result["parse_method"], "malformed": parse_result["malformed"],
                "raw_stage1_line": parse_result["raw_line"]}

    def run_staged_inference(self, problem_item, max_new_tokens_stage1=20, max_new_tokens_stage2=128):
        expr = problem_item["expression"]
        data_str = problem_item["data_str"]
        irrelevant_vars = problem_item.get("irrelevant_vars", [])
        problem_id = problem_item.get("id", "sample_001")
        problem_type = problem_item.get("type", "B")

        self._ensure_adapters_loaded()
        input_device = self.model.get_input_embeddings().weight.device

        self._activate(self._privacy_adapter_name)
        stage1_prompt = (
            f"<|start_header_id|>system<|end_header_id|>\\n{self.STAGE1_SYSTEM}\\n"
            f"<|eot_id|><|start_header_id|>user<|end_header_id|>\\n"
            f"Expression: {expr}\\nData: {data_str}<|eot_id|>"
            f"<|start_header_id|>assistant<|end_header_id|>\\n"
        )
        inputs1 = self.tokenizer(stage1_prompt, return_tensors="pt").to(input_device)
        with torch.no_grad():
            out1 = self.model.generate(**inputs1, max_new_tokens=max_new_tokens_stage1,
                                        do_sample=False, pad_token_id=self.tokenizer.pad_token_id)
        stage1_text = self.tokenizer.decode(out1[0][inputs1.input_ids.shape[1]:], skip_special_tokens=True)

        parse_result = self.parse_emitted_calls(stage1_text)
        telemetry_log = self.compute_dms_telemetry(problem_id, problem_type, parse_result, irrelevant_vars)

        self._activate(self._accuracy_adapter_name)
        accessed_vars = parse_result["accessed_vars"]
        kept_vars_str = ", ".join(accessed_vars) if accessed_vars else "NONE"
        stage2_prompt = (
            f"<|start_header_id|>system<|end_header_id|>\\n"
            f"You are a mathematical execution engine. Compute the final answer using ONLY the permitted variables.\\n"
            f"<|eot_id|><|start_header_id|>user<|end_header_id|>\\n"
            f"Expression: {expr}\\nPermitted Variables: {kept_vars_str}\\nData: {data_str}<|eot_id|>"
            f"<|start_header_id|>assistant<|end_header_id|>\\n"
        )
        inputs2 = self.tokenizer(stage2_prompt, return_tensors="pt").to(input_device)
        with torch.no_grad():
            out2 = self.model.generate(**inputs2, max_new_tokens=max_new_tokens_stage2,
                                        do_sample=False, pad_token_id=self.tokenizer.pad_token_id)
        stage2_answer = self.tokenizer.decode(out2[0][inputs2.input_ids.shape[1]:], skip_special_tokens=True)
        return telemetry_log, stage2_answer
'''

stub_file_path = os.path.join(raf_repo_dir, "staged_local_stub.py")
with open(stub_file_path, "w") as f:
    f.write(fixed_stub_code)
print(f"✅ Overwrote {stub_file_path} with fixed stub code")

from staged_local_stub import LocalStagedDecodingStub

# 2. Load Base Model in Evaluation Mode
eval_base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",   # shard across both GPUs
    torch_dtype=torch.bfloat16,
    attn_implementation="eager"
)

# 3. Initialize Stub pointing to Kaggle Adapter Paths
stub = LocalStagedDecodingStub(
    base_model=eval_base_model,
    tokenizer=tokenizer,
    privacy_adapter_path=privacy_adapter_path,
    accuracy_adapter_path=accuracy_adapter_path
)

# 4. Load Test Benchmark 
with open(benchmark_file, "r") as f:
    test_cases = [json.loads(line) for line in f]

telemetry_results = []
running_dms = {"A": [], "B": [], "C": []}

print(f"Starting Live Evaluation on {len(test_cases)} benchmark items...\n")
print(f"{'Item':<10} | {'Type':<6} | {'Accessed':<12} | {'Item DMS':<8} | {'Latency':<8} | {'Running DMS (A/B/C)':<20}")
print("-" * 75)

# 5. Live Inference Loop
for idx, item in enumerate(tqdm(test_cases, desc="Evaluating Benchmark")):
    p_id = item.get("id", f"item_{idx}")
    p_type = item.get("type", "B")

    start_t = time.time()
    telemetry_log, final_ans = stub.run_staged_inference(item)
    latency = time.time() - start_t

    dms_score = telemetry_log["dms"]
    running_dms[p_type].append(dms_score)

    telemetry_results.append({
        "problem_id": p_id,
        "telemetry": telemetry_log,
        "final_answer": final_ans,
        "latency_sec": round(latency, 2)
    })

    avg_a = sum(running_dms["A"]) / len(running_dms["A"]) if running_dms["A"] else 0.0
    avg_b = sum(running_dms["B"]) / len(running_dms["B"]) if running_dms["B"] else 0.0
    avg_c = sum(running_dms["C"]) / len(running_dms["C"]) if running_dms["C"] else 0.0

    if (idx + 1) % 2 == 0 or idx == 0 or (idx + 1) == len(test_cases):
        acc_str = ",".join(telemetry_log["accessed"]) if telemetry_log["accessed"] else "NONE"
        print(f"{p_id:<10} | {p_type:<6} | {acc_str:<12} | {dms_score:<8.2f} | {latency:<6.2f}s | A:{avg_a:.2f} B:{avg_b:.2f} C:{avg_c:.2f}")

# 6. Save Telemetry Artifact
final_log_file = os.path.join(OUTPUT_DIR, "staged_decoding_telemetry_final.jsonl")
with open(final_log_file, "w") as f:
    for entry in telemetry_results:
        f.write(json.dumps(entry) + "\n")

print(f"\nBenchmark complete! Standardized Oracle JSON logs saved to: {final_log_file}")

✅ Overwrote /kaggle/working/ReasoningAuthenticationFramework-RAF-/staged_local_stub.py with fixed stub code


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Starting Live Evaluation on 100 benchmark items...

Item       | Type   | Accessed     | Item DMS | Latency  | Running DMS (A/B/C) 
---------------------------------------------------------------------------


...

Loading weights:   0%|          | 0/128 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Loading weights:   0%|          | 0/128 [00:00<?, ?it/s]

LlamaForCausalLM LOAD REPORT from: /kaggle/working/staged_decoding/accuracy_adapter_final
Key                                                          | Status  | 
-------------------------------------------------------------+---------+-
model.layers.{0...31}.self_attn.q_proj.lora_B.privacy.weight | MISSING | 
model.layers.{0...31}.self_attn.q_proj.lora_A.privacy.weight | MISSING | 
model.layers.{0...31}.self_attn.v_proj.lora_B.privacy.weight | MISSING | 
model.layers.{0...31}.self_attn.v_proj.lora_A.privacy.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


type_b_00023 | B      | Y            | 0.97     | 6.68  s | A:0.00 B:0.97 C:0.00
type_c_00016 | C      | C,D,X        | 0.91     | 5.48  s | A:0.00 B:0.97 C:0.91
type_c_00028 | C      | C,Z,D        | 0.91     | 5.48  s | A:0.00 B:0.97 C:0.91
type_b_00018 | B      | A            | 0.97     | 4.78  s | A:0.00 B:0.97 C:0.91
type_b_00000 | B      | Y            | 0.97     | 4.81  s | A:0.00 B:0.97 C:0.91
type_a_00024 | A      | NONE         | 1.00     | 3.68  s | A:1.00 B:0.97 C:0.91
type_a_00022 | A      | NONE         | 1.00     | 3.74  s | A:1.00 B:0.97 C:0.91
type_c_00025 | C      | X,B,C        | 0.91     | 5.53  s | A:1.00 B:0.97 C:0.91
type_a_00014 | A      | NONE         | 1.00     | 3.75  s | A:1.00 B:0.97 C:0.91
type_b_00005 | B      | X            | 0.97     | 4.91  s | A:1.00 B:0.97 C:0.91
type_c_00038 | C      | B,A,D        | 0.91     | 5.88  s | A:1.00 B:0.97 C:0.91
type_b_00003 | B      | Z            | 0.97     | 5.02  s | A:1.00 B:0.97 C:0.91
type_c_00044 | C      | Z,B,

In [ ]:
import json

# Check what's actually saved inside each adapter's config
for path in [privacy_adapter_path, accuracy_adapter_path]:
    config_path = os.path.join(path, "adapter_config.json")
    with open(config_path) as f:
        cfg = json.load(f)
    print(path)
    print(" target_modules:", cfg.get("target_modules"))
    print(" peft_type:", cfg.get("peft_type"))
    print()

# List actual weight files present
!ls -la {privacy_adapter_path}
print("---")
!ls -la {accuracy_adapter_path}

In [ ]:
# 1. Clone RAF repository to Kaggle workspace
raf_repo_dir = "/kaggle/working/ReasoningAuthenticationFramework-RAF-"
if not os.path.exists(raf_repo_dir):
    !git clone https://github.com/ringerH/ReasoningAuthenticationFramework-RAF-.git {raf_repo_dir}

sys.path.append(raf_repo_dir)
from staged_local_stub import LocalStagedDecodingStub

with open(benchmark_file, "r") as f:
    test_cases = [json.loads(line) for line in f]
    

# 2. Load Base Model in Evaluation Mode
eval_base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    attn_implementation="eager"
)

# 3. Initialize Stub pointing to Kaggle Adapter Paths
stub = LocalStagedDecodingStub(
    base_model=eval_base_model,
    tokenizer=tokenizer,
    privacy_adapter_path=privacy_adapter_path,
    accuracy_adapter_path=accuracy_adapter_path
)

# 4. Load Test Benchmark
benchmark_file = os.path.join(raf_repo_dir, "data/test_sets/iccr_test_set.jsonl")
with open(benchmark_file, "r") as f:
    test_cases = [json.loads(line) for line in f]

telemetry_results = []
running_dms = {"A": [], "B": [], "C": []}

print(f"Starting Live Evaluation on {len(test_cases)} benchmark items...\n")
print(f"{'Item':<10} | {'Type':<6} | {'Accessed':<12} | {'Item DMS':<8} | {'Latency':<8} | {'Running DMS (A/B/C)':<20}")
print("-" * 75)

# 5. Live Inference Loop
for idx, item in enumerate(tqdm(test_cases, desc="Evaluating Benchmark")):
    p_id = item.get("id", f"item_{idx}")
    p_type = item.get("type", "B")
    
    start_t = time.time()
    telemetry_log, final_ans = stub.run_staged_inference(item)
    latency = time.time() - start_t
    
    dms_score = telemetry_log["dms"]
    running_dms[p_type].append(dms_score)
    
    telemetry_results.append({
        "problem_id": p_id,
        "telemetry": telemetry_log,
        "final_answer": final_ans,
        "latency_sec": round(latency, 2)
    })
    
    avg_a = sum(running_dms["A"]) / len(running_dms["A"]) if running_dms["A"] else 0.0
    avg_b = sum(running_dms["B"]) / len(running_dms["B"]) if running_dms["B"] else 0.0
    avg_c = sum(running_dms["C"]) / len(running_dms["C"]) if running_dms["C"] else 0.0
    
    # Print progress every 2 items
    if (idx + 1) % 2 == 0 or idx == 0 or (idx + 1) == len(test_cases):
        acc_str = ",".join(telemetry_log["accessed"]) if telemetry_log["accessed"] else "NONE"
        print(f"{p_id:<10} | {p_type:<6} | {acc_str:<12} | {dms_score:<8.2f} | {latency:<6.2f}s | A:{avg_a:.2f} B:{avg_b:.2f} C:{avg_c:.2f}")

# 6. Save Telemetry Artifact
final_log_file = os.path.join(OUTPUT_DIR, "staged_decoding_telemetry_final.jsonl")
with open(final_log_file, "w") as f:
    for entry in telemetry_results:
        f.write(json.dumps(entry) + "\n")

print(f"\nBenchmark complete! Standardized Oracle JSON logs saved to: {final_log_file}")


In [ ]:
with open(benchmark_file, "r") as f:
    test_cases = [json.loads(line) for line in f]

print("Total items:", len(test_cases))
print("Keys:", list(test_cases[0].keys()))
print(json.dumps(test_cases[0], indent=2))